# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
[FAIR² Mass Spectrometry Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa) via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant JSON-LD schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# Display a summary of metadata
meta = dataset.metadata
print(f"Dataset Name: {meta.name}")
print(f"Description: {meta.description}")
print(f"Authors: {getattr(meta, 'author', None)}")
print(f"Published: {getattr(meta, 'datePublished', None)}")
print(f"License: {getattr(meta, 'license', None)}")

## 2. Data Overview
Review available record sets and fields, including their `@id`s. All entities are referenced by their `@id` values.

In [ ]:
# Get all record set @ids
record_sets = dataset.record_sets
print(f"Number of record sets: {len(record_sets)}\n")

for rs in record_sets:
    print(f"Record Set @id: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {getattr(rs, 'description', None)}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.id}    (Name: {field.name} | Type: {field.data_type})")
    print("")

## 3. Data Extraction
Load data from selected record set(s) into a DataFrame for analysis. Record set and field access is always by `@id` as shown above.

In [ ]:
# Extract data from each record set into pandas DataFrames (by @id only)
dataframes = {}
for rs in record_sets:
    print(f"Loading records for RecordSet: {rs.id}")
    df = pd.DataFrame(dataset.records(record_set=rs.id))
    dataframes[rs.id] = df
    print(f"Fields (@id): {list(df.columns)}\n")

# For demonstration, let's select the first record set (usually the main data)
if len(record_sets) > 0:
    main_rs_id = record_sets[0].id
    display_cols = dataframes[main_rs_id].columns.tolist()
    print(f"Columns for main record set ({main_rs_id}): {display_cols}")
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)

This section demonstrates basic analysis steps such as filtering records based on a numeric field, normalizing a field, and grouping results. All columns accessed by their `@id`.

In [ ]:
# --- EDIT BELOW: Ensure field @ids below match those discovered in data overview! ---
record_set_id = main_rs_id  # use the main record set found earlier
# Example: select a likely numeric field id, update as appropriate.
possible_numeric_fields = [col for col in dataframes[record_set_id].columns if col.lower().find('count') != -1 or col.lower().find('abundance')!=-1 or col.lower().find('mw')!=-1]
print(f"Possible numeric fields: {possible_numeric_fields}")

# Use the first discovered numeric field (for demonstration, replace if needed)
numeric_field = possible_numeric_fields[0] if possible_numeric_fields else dataframes[record_set_id].columns[0]
print(f"Chosen numeric field: {numeric_field}\n")

# Filter for values above a threshold
threshold = 10
df = dataframes[record_set_id]
# Coerce to numeric if not already (ignore errors)
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold}: total {len(filtered_df)}\n")
display(filtered_df.head())

# Normalize field
if filtered_df[numeric_field].std() != 0:
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
else:
    filtered_df[f"{numeric_field}_normalized"] = filtered_df[numeric_field]
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Categorize/group by a field (e.g., description or categorical column if it exists)
group_field_candidates = [col for col in df.columns if col.lower().find('desc')!=-1 or col.lower().find('type')!=-1 or col.lower().find('category')!=-1]
group_field = group_field_candidates[0] if group_field_candidates else None
if group_field:
    print(f"Grouping by: {group_field}\n")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    display(grouped_df.head())
else:
    print('No suitable group field found for demonstration.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using standard plotting libraries (e.g., matplotlib, seaborn).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
plt.figure(figsize=(8,5))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True, color='skyblue')
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# If grouping field exists, do a barplot of means
if group_field:
    plt.figure(figsize=(10,5))
    plot_df = grouped_df.sort_values(numeric_field, ascending=False)
    sns.barplot(data=plot_df, x=group_field, y=numeric_field, palette='viridis')
    plt.title(f'Mean {numeric_field} by {group_field}')
    plt.xlabel(group_field)
    plt.ylabel(f'Mean {numeric_field}')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
This notebook guided you through exploring the FAIR² human mast cell mass spectrometry dataset with the `mlcroissant` library.

- Loaded dataset metadata and examined key properties.
- Explored available record sets and their field `@id`s.
- Loaded records into DataFrames and previewed structure.
- Demonstrated filtering and normalization based on numeric columns (`@id` reference), grouped by other categorical variables if present.
- Visualized the data for further insight.

For further work, more advanced filtering, feature engineering, and model-building can be conducted, always using Croissant `@id` references for stable access.